# Uçuş Gecikmesi Tahmini — 0. Veriye ilk bakış
**Veri:** Kaggle — *US Flight Delays 2025 (BTS On-Time Performance)*
https://www.kaggle.com/datasets/a7madmostafa/us-flight-delays-2025-bts-on-time-performance

Bu notebook veriyi **değiştirmez**; sadece indirir, Drive'a kaydeder ve yapısına bakar.

**Nasıl çalıştırılır:** Çalışma zamanı → Tümünü çalıştır. Drive izni sorarsa tüm kutuları işaretle.
İlk çalıştırmada indirme ve Drive'a kopyalama birkaç dakika sürebilir; sonraki çalıştırmalarda veri Drive'dan okunur.
Bitince **Dosya → İndir → .ipynb** ile indirip gönder.

## 0. Kurulum ve klasörler

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub', 'duckdb'], check=False)

import shutil
from pathlib import Path

import duckdb
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/UcusGecikme')
    IN_COLAB = True
except ImportError:
    BASE = Path('.')
    IN_COLAB = False

RAW = BASE / 'raw' / 'flights_2025'
RAW.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
print('Veri klasörü:', RAW)

Mounted at /content/drive
Veri klasörü: /content/drive/MyDrive/UcusGecikme/raw/flights_2025


## 1. İndirme (sadece Drive'da yoksa)

In [2]:
def data_files(folder):
    return sorted([p for p in folder.rglob('*') if p.suffix.lower() in ('.csv', '.parquet')])

if not data_files(RAW):
    import kagglehub
    try:
        src = Path(kagglehub.dataset_download('a7madmostafa/us-flight-delays-2025-bts-on-time-performance'))
    except Exception as e:
        print('İndirme hatası:', e)
        print('Kaggle giriş istiyorsa: aşağıdaki satırı ayrı bir hücrede çalıştır, açılan kutuya Kaggle kullanıcı adını ve API anahtarını gir:')
        print('    import kagglehub; kagglehub.login()')
        print('(API anahtarı: kaggle.com → sağ üstte profil → Settings → API → Create New Token)')
        raise
    print('İndirildi:', src)
    for f in data_files(src):
        target = RAW / f.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, target)
        print('  Drive\'a kopyalandı:', target.name)
else:
    print('Veri zaten Drive\'da, indirme atlandı.')

files = data_files(RAW)
assert files, 'Veri dosyası bulunamadı.'
print(f'\n{len(files)} dosya, toplam {sum(f.stat().st_size for f in files) / 1e9:.2f} GB')
for f in files:
    print(f'  {f.name:60s} {f.stat().st_size / 1e6:8.1f} MB')

100%|██████████| 244M/244M [00:02<00:00, 108MB/s]

Extracting files...


İndirildi: /root/.cache/kagglehub/datasets/a7madmostafa/us-flight-delays-2025-bts-on-time-performance/versions/1
  Drive'a kopyalandı: flight_delays_2025_01.csv
  Drive'a kopyalandı: flight_delays_2025_02.csv
  Drive'a kopyalandı: flight_delays_2025_03.csv
  Drive'a kopyalandı: flight_delays_2025_04.csv
  Drive'a kopyalandı: flight_delays_2025_05.csv
  Drive'a kopyalandı: flight_delays_2025_06.csv
  Drive'a kopyalandı: flight_delays_2025_07.csv
  Drive'a kopyalandı: flight_delays_2025_08.csv
  Drive'a kopyalandı: flight_delays_2025_09.csv
  Drive'a kopyalandı: flight_delays_2025_10.csv
  Drive'a kopyalandı: flight_delays_2025_11.csv
  Drive'a kopyalandı: flight_delays_2025_12.csv

12 dosya, toplam 1.41 GB
  flight_delays_2025_01.csv                                       107.4 MB
  flight_delays_2025_02.csv                                       101.2 MB
  flight_delays_2025_03.csv                                       120.4 MB
  flight_delays_2025_04.csv                                 

## 2. Tek bir aya yakından bakış
Önce ilk dosyayı (bir ay) açıp sütunları tanıyalım.

In [3]:
first = files[0]
m = pd.read_parquet(first) if first.suffix.lower() == '.parquet' else pd.read_csv(first, low_memory=False)
print(f'{first.name}: {len(m):,} satır, {m.shape[1]} sütun\n')
print('İlk 2 satır (sütunlar alt alta):')
print(m.head(2).T.to_string())

flight_delays_2025_01.csv: 539,747 satır, 40 sütun

İlk 2 satır (sütunlar alt alta):
                                               0                1
Year                                        2025             2025
Quarter                                        1                1
Month                                          1                1
DayofMonth                                     1                2
DayOfWeek                                      3                4
FlightDate                            2025-01-01       2025-01-02
Reporting_Airline                             AA               AA
Flight_Number_Reporting_Airline                1                1
Origin                                       JFK              JFK
OriginCityName                      New York, NY     New York, NY
OriginState                                   NY               NY
Dest                                         LAX              LAX
DestCityName                     Los Angeles, CA  Los Ang

In [4]:
summary = pd.DataFrame({
    'tip': m.dtypes.astype(str),
    'farklı_değer': m.nunique(),
    'eksik': m.isna().sum(),
    'eksik_%': (m.isna().mean() * 100).round(1),
})
print(summary.to_string())

                                     tip  farklı_değer   eksik  eksik_%
Year                               int64             1       0      0.0
Quarter                            int64             1       0      0.0
Month                              int64             1       0      0.0
DayofMonth                         int64            31       0      0.0
DayOfWeek                          int64             7       0      0.0
FlightDate                        object            31       0      0.0
Reporting_Airline                 object            14       0      0.0
Flight_Number_Reporting_Airline    int64          6390       0      0.0
Origin                            object           329       0      0.0
OriginCityName                    object           323       0      0.0
OriginState                       object            52       0      0.0
Dest                              object           329       0      0.0
DestCityName                      object           323       0  

## 3. Sütunları gruplayalım: uçuştan ÖNCE bilinenler / SONRA bilinenler
Sonra bilinen sütunlar modele girerse **sızıntı** olur (model cevabı görmüş olur).

In [5]:
cols = list(m.columns)
lower = {c.lower(): c for c in cols}

def find(*keys):
    """Adında verilen parçalardan biri geçen sütunlar (büyük/küçük harf fark etmez)."""
    return [c for c in cols if any(k in c.lower() for k in keys)]

after_keys = ('dep_time', 'deptime', 'arr_time', 'arrtime', 'wheels', 'taxi', 'actual_elapsed', 'air_time',
              'airtime', 'delay', 'del15', 'cancel', 'divert', 'dep_delay', 'arr_delay', 'delaygroup', 'delay_group')
after = [c for c in find(*after_keys) if 'crs' not in c.lower()]
before = [c for c in cols if c not in after]
print('Uçuştan ÖNCE bilinen (modelde kullanılabilir) sütunlar:')
print('  ', before)
print('\nUçuştan SONRA bilinen (target veya sızıntı) sütunlar:')
print('  ', after)

Uçuştan ÖNCE bilinen (modelde kullanılabilir) sütunlar:
   ['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Origin', 'OriginCityName', 'OriginState', 'Dest', 'DestCityName', 'DestState', 'CRSDepTime', 'CRSArrTime', 'CRSElapsedTime', 'ActualElapsedTime', 'Distance']

Uçuştan SONRA bilinen (target veya sızıntı) sütunlar:
   ['DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'Cancelled', 'CancellationCode', 'Diverted', 'AirTime', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']


## 4. Target: 15 dakikadan fazla gecikme

In [6]:
def pick(*names):
    for n in names:
        if n.lower() in lower:
            return lower[n.lower()]
    return None

ARR15 = pick('ARR_DEL15', 'ArrDel15', 'arr_del15')
ARRD = pick('ARR_DELAY', 'ArrDelay', 'arr_delay', 'ARR_DELAY_NEW')
CANC = pick('CANCELLED', 'Cancelled', 'cancelled')
DIV = pick('DIVERTED', 'Diverted', 'diverted')
print('Bulunan sütunlar → varış gecikmesi 15+:', ARR15, '| varış gecikmesi (dk):', ARRD, '| iptal:', CANC, '| yön değiştirme:', DIV)

if ARR15:
    vc = m[ARR15].value_counts(dropna=False)
    print(f'\n{ARR15} dağılımı (bu ay):')
    print(pd.concat([vc, (vc / len(m) * 100).round(1)], axis=1, keys=['adet', '%']).to_string())
if CANC:
    print(f'\nİptal edilen uçuş: {int(m[CANC].sum()):,} (%{m[CANC].mean() * 100:.1f})')
if DIV:
    print(f'Başka havalimanına inen: {int(m[DIV].sum()):,} (%{m[DIV].mean() * 100:.1f})')
if ARR15 and CANC:
    print(f'İptal edilen uçuşlarda {ARR15} boş mu? → boş oranı: %{m.loc[m[CANC] == 1, ARR15].isna().mean() * 100:.0f}')
if ARRD:
    print(f'\nVarış gecikmesi (dk) özeti:')
    print(m[ARRD].describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())

Bulunan sütunlar → varış gecikmesi 15+: ArrDel15 | varış gecikmesi (dk): ArrDelay | iptal: Cancelled | yön değiştirme: Diverted

ArrDel15 dağılımı (bu ay):
            adet     %
ArrDel15              
0.0       424139  78.6
1.0        98130  18.2
NaN        17478   3.2

İptal edilen uçuş: 16,312 (%3.0)
Başka havalimanına inen: 1,166 (%0.2)
İptal edilen uçuşlarda ArrDel15 boş mu? → boş oranı: %100

Varış gecikmesi (dk) özeti:
count    522269.0
mean          3.8
std          56.9
min         -87.0
25%         -18.0
50%          -8.0
75%           7.0
90%          37.0
99%         205.0
max        3282.0


## 5. Tüm yıl: aylara göre uçuş sayısı ve gecikme oranı
7 milyon satırı belleğe almadan DuckDB ile dosyaların üzerinden sayıyoruz.

In [7]:
reader = 'read_parquet' if files[0].suffix.lower() == '.parquet' else 'read_csv_auto'
opts = '' if reader == 'read_parquet' else ', union_by_name=true, sample_size=-1'
glob = str(RAW / '**' / f'*{files[0].suffix}')
con = duckdb.connect()
src_sql = f"{reader}('{glob}'{opts}, filename=true)"

total = con.sql(f'SELECT count(*) FROM {src_sql}').fetchone()[0]
print(f'TOPLAM UÇUŞ (tüm dosyalar): {total:,}\n')

agg = f', round(avg("{ARR15}") * 100, 1) AS gecikme_15dk_yuzde' if ARR15 else ''
canc = f', round(avg("{CANC}") * 100, 1) AS iptal_yuzde' if CANC else ''
print(con.sql(f'''SELECT regexp_extract(filename, '[^/]+$') AS dosya, count(*) AS ucus{agg}{canc}
                  FROM {src_sql} GROUP BY 1 ORDER BY 1''').df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TOPLAM UÇUŞ (tüm dosyalar): 7,001,619



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                    dosya   ucus  gecikme_15dk_yuzde  iptal_yuzde
flight_delays_2025_01.csv 539747                18.8          3.0
flight_delays_2025_02.csv 504884                20.8          1.5
flight_delays_2025_03.csv 600872                19.6          1.2
flight_delays_2025_04.csv 583950                19.7          0.8
flight_delays_2025_05.csv 605648                23.6          1.0
flight_delays_2025_06.csv 611575                28.3          1.6
flight_delays_2025_07.csv 631428                28.9          2.5
flight_delays_2025_08.csv 602378                22.6          1.1
flight_delays_2025_09.csv 562439                16.6          0.5
flight_delays_2025_10.csv 605844                20.3          0.5
flight_delays_2025_11.csv 570550                20.6          2.5
flight_delays_2025_12.csv 582304                26.8          1.5


## 6. Havayolları ve havalimanları

In [8]:
CARR = pick('OP_UNIQUE_CARRIER', 'Reporting_Airline', 'OP_CARRIER', 'IATA_CODE_Reporting_Airline', 'Marketing_Airline_Network', 'AIRLINE', 'carrier')
ORIG = pick('ORIGIN', 'Origin', 'origin')
DEST = pick('DEST', 'Dest', 'dest')
print('Havayolu sütunu:', CARR, '| kalkış:', ORIG, '| varış:', DEST)

if CARR:
    print(f'\nHavayolu sayısı (tüm yıl):', con.sql(f'SELECT count(DISTINCT "{CARR}") FROM {src_sql}').fetchone()[0])
    q = f'SELECT "{CARR}" AS havayolu, count(*) AS ucus' + (f', round(avg("{ARR15}") * 100, 1) AS gecikme_yuzde' if ARR15 else '')
    print(con.sql(f'{q} FROM {src_sql} GROUP BY 1 ORDER BY ucus DESC').df().to_string(index=False))
if ORIG and DEST:
    n_airports = con.sql(f'SELECT count(DISTINCT x) FROM (SELECT "{ORIG}" x FROM {src_sql} UNION SELECT "{DEST}" FROM {src_sql})').fetchone()[0]
    print(f'\nFarklı havalimanı: {n_airports}')
    q = f'SELECT "{ORIG}" AS kalkis, count(*) AS ucus' + (f', round(avg("{ARR15}") * 100, 1) AS gecikme_yuzde' if ARR15 else '')
    print('\nEn yoğun 15 kalkış havalimanı:')
    print(con.sql(f'{q} FROM {src_sql} GROUP BY 1 ORDER BY ucus DESC LIMIT 15').df().to_string(index=False))

Havayolu sütunu: Reporting_Airline | kalkış: Origin | varış: Dest


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Havayolu sayısı (tüm yıl): 14


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

havayolu    ucus  gecikme_yuzde
      WN 1391885           21.4
      DL 1026332           19.8
      AA  973653           25.7
      OO  839821           20.7
      UA  795271           21.5
      YX  346036           21.5
      MQ  299322           21.0
      OH  248735           27.5
      AS  245588           23.1
      B6  231413           26.2
      F9  198065           27.9
      NK  194515           21.4
      G4  130899           25.0
      HA   80084           17.3


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Farklı havalimanı: 352

En yoğun 15 kalkış havalimanı:


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

kalkis   ucus  gecikme_yuzde
   ORD 327028           26.5
   DEN 317686           25.7
   DFW 315854           28.7
   ATL 314433           24.0
   PHX 196756           19.6
   CLT 195425           24.7
   LAX 190472           17.6
   LAS 183803           20.8
   SEA 164835           22.2
   MCO 160048           24.5
   SFO 144495           19.8
   BOS 142859           24.2
   DCA 142506           27.1
   LGA 135725           26.4
   DTW 122533           21.8


## 7. Kirlilik kontrolleri

In [9]:
dup = m.duplicated().sum()
print(f'Bu ayda tamamen aynı satır: {dup:,}')
FN = pick('OP_CARRIER_FL_NUM', 'Flight_Number_Reporting_Airline', 'FL_NUM', 'flight_number')
DATE = pick('FL_DATE', 'FlightDate', 'fl_date', 'date')
if FN and CARR and DATE and ORIG:
    key = [DATE, CARR, FN, ORIG]
    print(f'Aynı tarih + havayolu + uçuş no + kalkış ile birden fazla satır: {m.duplicated(key).sum():,}')
CRS = pick('CRS_DEP_TIME', 'CRSDepTime', 'crs_dep_time')
if CRS:
    print(f'\nPlanlanan kalkış saati ({CRS}) örnek değerler: {[int(x) for x in sorted(m[CRS].dropna().unique())[:10]]}')
    print('  (HHMM biçiminde mi, 2400 gibi garip değerler var mı?)', 'max =', m[CRS].max())
TAIL = pick('TAIL_NUM', 'Tail_Number', 'tail_number')
if TAIL:
    print(f'\nKuyruk numarası ({TAIL}) eksik: %{m[TAIL].isna().mean() * 100:.1f}  — uçak bilgisiyle birleştirmede kullanılabilir')

Bu ayda tamamen aynı satır: 0
Aynı tarih + havayolu + uçuş no + kalkış ile birden fazla satır: 0

Planlanan kalkış saati (CRSDepTime) örnek değerler: [5, 10, 17, 20, 21, 22, 23, 25, 26, 27]
  (HHMM biçiminde mi, 2400 gibi garip değerler var mı?) max = 2359


## Bitti
**Dosya → İndir → .ipynb** ile indirip gönder. Sonraki adım: hava durumu verisini (NOAA) indirip havalimanlarıyla eşleştirmek.